In [2]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [6]:
print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

In [4]:

# Task 1: Scatter plot — Europe, 2002 vs 2007

import pandas as pd
import plotly.express as px

df = px.data.gapminder()

# Filter Europe and the two selected years
europe_df = df.loc[
    (df["continent"] == "Europe") &
    (df["year"].isin([2002, 2007]))
].copy()

# Create scatter plot
fig = px.scatter(
    europe_df,
    x="gdpPercap",
    y="lifeExp",
    color="year",
    color_discrete_map={
        2002: "#9CA3AF",
        2007: "#2563EB"
    },
    hover_name="country",
    hover_data={
        "year": True,
        "gdpPercap": ":,.0f",
        "lifeExp": ":.1f"
    },
    category_orders={"year": [2002, 2007]},
    title="<b>Europe: GDP and life expectancy generally rose from 2002 to 2007</b>",
    labels={
        "gdpPercap": "GDP per capita (US$)",
        "lifeExp": "Life expectancy (years)",
        "year": "Year"
    }
)

# Logarithmic scale for GDP per capita
fig.update_xaxes(type="log")

# Connect each country's two observations with a light grey line
for country in europe_df["country"].unique():
    country_data = (
        europe_df[europe_df["country"] == country]
        .sort_values("year")
    )

    if len(country_data) == 2:
        fig.add_trace(
            px.line(
                country_data,
                x="gdpPercap",
                y="lifeExp"
            ).data[0].update(
                line=dict(color="#D1D5DB", width=1),
                showlegend=False,
                hoverinfo="skip"
            )
        )

# Keep the year-coloured points above the connecting lines
fig.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=9, line=dict(width=0.5, color="white"))
)

fig.update_layout(
    template="plotly_white",
    plot_bgcolor="white",
    title_x=0.02,
    margin=dict(l=70, r=30, t=100, b=60),
    legend_title_text=""
)

fig.update_xaxes(showgrid=False, showline=False)
fig.update_yaxes(
    showgrid=True,
    gridcolor="#EEEEEE",
    showline=False
)

fig.show()


In [5]:

# Task 2: Bubble chart — highlight Asia in 2007

import pandas as pd
import plotly.express as px

df = px.data.gapminder()

# Use the full 2007 dataset
df_2007 = df[df["year"] == 2007].copy()

# Highlight Asia; grey out all other continents
df_2007["Group"] = df_2007["continent"].apply(
    lambda continent: (
        "Asia (highlighted)"
        if continent == "Asia"
        else "Other continents"
    )
)

fig = px.scatter(
    df_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    size_max=35,
    color="Group",
    color_discrete_map={
        "Asia (highlighted)": "#2563EB",
        "Other continents": "#D1D5DB"
    },
    hover_name="country",
    hover_data={
        "continent": True,
        "gdpPercap": ":,.0f",
        "lifeExp": ":.1f",
        "pop": ":,.0f"
    },
    title="<b>Asia's countries span a wide range of income and longevity</b>",
    labels={
        "gdpPercap": "GDP per capita (US$)",
        "lifeExp": "Life expectancy (years)",
        "pop": "Population",
        "Group": ""
    }
)

# Log scale for GDP per capita
fig.update_xaxes(type="log")

# Annotation explaining the highlighted group's story
fig.add_annotation(
    x=0.98,
    y=0.05,
    xref="paper",
    yref="paper",
    text="Blue = Asia<br>Bubble size = population",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    bgcolor="white",
    bordercolor="#D1D5DB",
    borderwidth=1,
    font=dict(size=11, color="#374151")
)

fig.update_layout(
    template="plotly_white",
    plot_bgcolor="white",
    title_x=0.02,
    margin=dict(l=70, r=30, t=100, b=60),
    legend_title_text=""
)

fig.update_xaxes(showgrid=False, showline=False)
fig.update_yaxes(
    showgrid=True,
    gridcolor="#EEEEEE",
    showline=False
)

fig.show()
